# Roboto Origin: separate unloaded walking skill

Use a GPU runtime and the authorized group Google Drive. This task is empty-handed walking only. Manipulation, loaded walking and the supervisor are not integrated. The original two-box task remains separate.

The setup may take several minutes. Another training round is disabled by default; change RUN_TRAINING when ready. Checkpoints and logs are copied to Drive with SHA-256 checks. Resume uses the latest backed-up walking checkpoint, including interrupted runs with a saved checkpoint.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
"""Colab recovery run; downloads its backup immediately after training finishes.

EULA acceptance was explicitly authorized by Zhao-Yu in this conversation.
This script uses only the group's public source and its own generated files.
"""

import os
import re
import subprocess
import sys
import time
from pathlib import Path

PYTHON = '/content/rpo-venv/bin/python'
LAB = Path('/content/IsaacLab')
REPO = Path('/content/LearningRL')
REVISION = '157893d06453dd77e60f0db627fa7b80e6b07817'


def checked(args, cwd=None, timeout=600):
    p = subprocess.run(args, cwd=cwd, text=True, stdout=subprocess.PIPE,
                       stderr=subprocess.STDOUT, timeout=timeout)
    print(p.stdout[-3500:], flush=True)
    p.check_returncode()


checked([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
if not Path(PYTHON).is_file():
    checked(['uv', 'venv', '--python', '3.12', '/content/rpo-venv'])
if not LAB.exists():
    checked(['git', 'clone', '--depth', '1', '--branch', 'v3.0.0-beta2.patch1',
             'https://github.com/isaac-sim/IsaacLab.git', str(LAB)])
if not REPO.exists():
    checked(['git', 'clone', '--branch', 'student/zhaoyu-lu-HMD', '--single-branch',
             'https://github.com/yuki-taiwanese/LearningRL.git', str(REPO)])
checked(['git', '-C', str(REPO), 'checkout', REVISION])
override = Path('/content/rpo-overrides.txt')
override.write_text('coverage==7.4.4\npackaging==26.0\n')
args = ['uv', 'pip', 'install', '--python', PYTHON, '--extra-index-url',
        'https://pypi.nvidia.com', '--index-strategy', 'unsafe-best-match',
        '--prerelease', 'allow', '--override', str(override),
        'isaacsim[all,extscache]==6.0.1.0', 'wandb', 'ruff']
for package in ['isaaclab', 'isaaclab_assets', 'isaaclab_contrib',
                'isaaclab_newton[all]', 'isaaclab_physx[newton]', 'isaaclab_rl[rsl-rl]',
                'isaaclab_tasks', 'isaaclab_ov', 'isaaclab_ovphysx',
                'isaaclab_ppisp', 'isaaclab_visualizers[kit]']:
    name, _, extras = package.partition('[')
    args += ['-e', str(LAB / 'source' / name) + ('[' + extras if extras else '')]
args += ['-e', str(REPO / 'robot_description'), '-e', str(REPO / 'LearningRLlab')]
print('Installing the matched simulation environment...', flush=True)
checked(args, timeout=900)

asset = REPO / 'robot_description/vendor/rpo_description'
asset.parent.mkdir(parents=True, exist_ok=True)
if not asset.exists():
    checked(['git', 'clone', 'https://github.com/Roboparty/rpo_description.git', str(asset)])
checked(['git', '-C', str(asset), 'checkout', '37aac9ca665e92731444a1618320078e7ba21569'])

env = os.environ.copy()
env.update(OMNI_KIT_ACCEPT_EULA='YES', ISAACLAB_PATH=str(LAB), WANDB_MODE='disabled',
           MPLBACKEND='Agg', PYTHONUNBUFFERED='1')
model_path = Path('/content/rpo-converted-path.txt')
if not model_path.exists():
    # Import with Kit once; training itself never uses a Vulkan renderer.
    conversion = r'''
import os
from pathlib import Path
from isaacsim import SimulationApp
app = SimulationApp({'headless': True})
from isaaclab.sim.converters import UrdfConverter
from robot_description.robots.roboparty_rpo import RPO_CFG
from robot_description.tools.usd_inertia_fix import fix_principal_axes
loader = UrdfConverter(RPO_CFG.spawn)
fix_principal_axes(loader.usd_path)
Path('/content/rpo-converted-path.txt').write_text(loader.usd_path)
print('CONVERTED_AND_INERTIA_FIXED:', loader.usd_path, flush=True)
# The converter has synchronously saved all layers. Avoid Colab's Kit shutdown crash.
os._exit(0)
'''
    with Path('/content/rpo-conversion.log').open('w') as log:
        p = subprocess.run([PYTHON, '-c', conversion], env=env, stdout=log,
                           stderr=subprocess.STDOUT, timeout=240)
    print(Path('/content/rpo-conversion.log').read_text()[-3500:], flush=True)
    assert model_path.exists(), 'Robot conversion did not complete; inspect rpo-conversion.log'

env['LEARNINGRL_RPO_USD'] = model_path.read_text().strip()
assert Path(env['LEARNINGRL_RPO_USD']).is_file()


In [ ]:
"""Colab round runner: finite jobs, live progress, periodic verified Drive copies."""

import hashlib
import json
import os
import re
import shutil
import subprocess
import time
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo


def run_round(task, experiment, role, iterations, num_envs=256, resume_run=None,
              checkpoint=None, source_commit=None, max_seconds=2700, hydra_overrides=()):
    python = '/content/rpo-venv/bin/python'
    repo = Path('/content/LearningRL')
    lab = Path('/content/IsaacLab')
    drive = Path('/content/drive/MyDrive/LearningRL/zhaoyu-lu-HMD')
    assert Path('/content/drive/MyDrive').is_dir(), 'Mount the authorized group Drive first'
    assert Path(python).is_file(), 'Simulation environment needs recovery first'
    model = Path('/content/rpo-converted-path.txt').read_text().strip()
    assert Path(model).is_file()
    actual_commit = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
    if source_commit:
        assert actual_commit == source_commit, (actual_commit, source_commit)
    stamp = datetime.now(ZoneInfo('America/Chicago')).strftime('%Y-%m-%d_%H%M%S')
    name = role + '_' + stamp
    backup = drive / 'rounds' / name
    backup.mkdir(parents=True, exist_ok=False)
    local_log = Path('/content') / (name + '.log')
    env = os.environ.copy()
    env.update(OMNI_KIT_ACCEPT_EULA='YES', ISAACLAB_PATH=str(lab), MPLBACKEND='Agg',
               WANDB_MODE='disabled', PYTHONUNBUFFERED='1', LEARNINGRL_RPO_USD=model)
    args = [python, str(repo / 'LearningRLlab/scripts/train.py'), '--task', task,
            '--num_envs', str(num_envs), '--max_iterations', str(iterations),
            '--viz', 'none', '--logger', 'tensorboard', '--run_name', name,
            'physics=newton_mjwarp', 'agent.save_interval=50']
    args += list(hydra_overrides)
    if resume_run:
        assert checkpoint
        args += ['--resume', '--load_run', resume_run, '--checkpoint', checkpoint]
    metadata = dict(task=task, role=role, requested_additional_updates=iterations,
                    num_envs=num_envs, source_commit=actual_commit, resume_run=resume_run,
                    resume_checkpoint=checkpoint, hydra_overrides=list(hydra_overrides),
                    status='running', name=name)
    (backup / 'round.json').write_text(json.dumps(metadata, indent=2))
    stable = {}
    copied = {}

    def sync(run, final=False):
        if run is None:
            return
        for path in run.glob('model_*.pt'):
            signature = (path.stat().st_size, path.stat().st_mtime_ns)
            if not final and stable.get(path.name) != signature:
                stable[path.name] = signature
                continue
            if signature[0] == 0 or copied.get(path.name) == signature:
                continue
            digest = hashlib.sha256(path.read_bytes()).hexdigest()
            temp = backup / (path.name + '.partial')
            destination = backup / path.name
            shutil.copy2(path, temp)
            if (path.stat().st_size, path.stat().st_mtime_ns) != signature:
                continue
            assert hashlib.sha256(temp.read_bytes()).hexdigest() == digest
            if destination.exists():
                assert hashlib.sha256(destination.read_bytes()).hexdigest() == digest
                temp.unlink()
            else:
                temp.replace(destination)
            copied[path.name] = signature
            print('DRIVE_CHECKPOINT_VERIFIED:', path.name, signature[0], flush=True)

    started = time.monotonic()
    run = None
    last_printed = -1
    with local_log.open('w') as log:
        process = subprocess.Popen(args, cwd=lab, env=env, stdout=log, stderr=subprocess.STDOUT)
        while process.poll() is None:
            time.sleep(10)
            candidates = list((lab / 'logs/rsl_rl' / experiment).glob('*_' + name))
            if candidates:
                run = sorted(candidates)[-1]
            sync(run)
            text = local_log.read_text(errors='replace')
            updates = re.findall(r'Learning iteration\s+(\d+)/(\d+)', text)
            count = len(updates)
            if count >= last_printed + 20:
                last_printed = count
                print(role, 'updates recorded:', count, '/', iterations, flush=True)
            if time.monotonic() - started > max_seconds:
                process.terminate()
                process.wait(timeout=20)
                metadata['status'] = 'compute_limit'
                break
    sync(run, final=True)
    text = local_log.read_text(errors='replace')
    rows = []
    for block in text.split('Learning iteration ')[1:]:
        match = re.match(r'(\d+)/(\d+)', block)
        if not match:
            continue
        row = {'iteration': int(match[1])}
        for line in block.splitlines():
            if ':' in line:
                key, value = line.rsplit(':', 1)
                try:
                    row[key.strip()] = float(value.strip())
                except ValueError:
                    pass
        rows.append(row)
    metadata.update(exit_code=process.returncode, recorded_updates=len(rows),
                    elapsed_seconds=round(time.monotonic() - started, 1),
                    checkpoints=list(copied), run_directory=str(run),
                    task_success_established=False, independent_evaluation_completed=False)
    metadata['status'] = 'completed' if process.returncode == 0 else metadata.get('status', 'failed')
    if process.returncode != 0 and metadata['status'] == 'running':
        metadata['status'] = 'failed'
    for key in ['Mean reward', 'Mean episode length', 'Metrics/base_velocity/stage',
                'Metrics/base_velocity/success', 'Metrics/base_velocity/error_vel_xy',
                'Metrics/base_velocity/error_vel_yaw']:
        values = [r[key] for r in rows if key in r]
        if values:
            metadata[key] = {'first': values[0], 'last': values[-1], 'max': max(values)}
    (backup / 'metrics.json').write_text(json.dumps(rows, indent=2))
    (backup / 'round.json').write_text(json.dumps(metadata, indent=2))
    shutil.copy2(local_log, backup / 'training.log')
    if run:
        for relative in ['params', 'git']:
            source = run / relative
            if source.is_dir():
                shutil.copytree(source, backup / relative, dirs_exist_ok=True)
    print('ROUND_RESULT:', json.dumps(metadata, indent=2), flush=True)
    if process.returncode != 0:
        print(text[-10000:], flush=True)
    assert process.returncode == 0, 'Training failed; preserved log and any checkpoints in Drive'
    assert len(rows) == iterations, 'Unexpected update count; inspect training.log'
    assert copied, 'No checkpoint was backed up'
    return metadata


In [ ]:
# Opt in when you want another finite training round.
RUN_TRAINING = False
ADDITIONAL_UPDATES = 500
if RUN_TRAINING:
    import shutil
    backups = Path('/content/drive/MyDrive/LearningRL/zhaoyu-lu-HMD/rounds')
    candidates = sorted([p for pattern in ('flat_walk_*', 'walk_tracking_*') for p in backups.glob(pattern) if list(p.glob('model_*.pt'))], key=lambda p:tuple(p.name.split('_')[-2:]))
    kwargs = {}
    if candidates:
        saved = candidates[-1]
        checkpoint = max(saved.glob('model_*.pt'), key=lambda p:int(p.stem.split('_')[-1]))
        restored = Path('/content/IsaacLab/logs/rsl_rl/learningrl_rpo_flat_walk') / ('restored_' + saved.name)
        restored.mkdir(parents=True, exist_ok=True)
        shutil.copy2(checkpoint, restored/checkpoint.name)
        kwargs = dict(resume_run=restored.name, checkpoint=checkpoint.name)
        print('RESUMING:', checkpoint)
    tracking_overrides = ['env.rewards.track_lin_vel_xy.weight=3.0', 'env.rewards.track_lin_vel_xy.params.std=0.15', 'env.rewards.track_ang_vel_z.weight=1.5', 'env.rewards.track_ang_vel_z.params.std=0.25']
    walk_round = run_round('LearningRL-RPO-FlatWalk-v0', 'learningrl_rpo_flat_walk', 'walk_tracking', ADDITIONAL_UPDATES, 256, source_commit=REVISION, hydra_overrides=tracking_overrides, **kwargs)
else:
    print('Training is off. Set RUN_TRAINING=True when ready for another round.')


## Results
Inspect `My Drive/LearningRL/zhaoyu-lu-HMD/rounds`. Higher reward is not proof of walking or box-transfer success. Keep independent trials, falls, command-tracking errors and all outcomes. Do not load a box-transfer checkpoint into the walking task: their observation dimensions differ.


## Optional independent walking evaluation
Five 10-second nominal trials. Reports first termination and velocity error. No model updates.


In [ ]:
EVALUATION_SOURCE = '"""Five independent nominal 10-second walking trials, using a frozen policy."""\nimport argparse\nimport importlib.metadata\nimport json\nimport sys\nfrom pathlib import Path\n\nimport warp as wp\nwp.config.enable_backward = False\nimport gymnasium as gym\nimport torch\nfrom rsl_rl.runners import OnPolicyRunner\nimport LearningRLlab.tasks\nfrom isaaclab_rl.rsl_rl import RslRlVecEnvWrapper, handle_deprecated_rsl_rl_cfg\nfrom isaaclab_tasks.utils import add_launcher_args, launch_simulation, setup_preset_cli\nfrom isaaclab_tasks.utils.hydra import hydra_task_config\n\nparser = argparse.ArgumentParser()\nparser.add_argument(\'--task\', default=\'LearningRL-RPO-FlatWalk-v0-PLAY\')\nparser.add_argument(\'--checkpoint\', required=True)\nparser.add_argument(\'--output\', required=True)\nadd_launcher_args(parser)\nargs, remaining = setup_preset_cli(parser)\nsys.argv = [sys.argv[0]] + remaining\n\n\n@hydra_task_config(args.task, \'rsl_rl_cfg_entry_point\')\ndef main(cfg, agent_cfg):\n    agent_cfg = handle_deprecated_rsl_rl_cfg(agent_cfg, importlib.metadata.version(\'rsl-rl-lib\'))\n    cfg.scene.num_envs = 5\n    cfg.seed = 20261116\n    cfg.observations.policy.enable_corruption = False\n    cfg.commands.base_velocity.resampling_time_range = (1000., 1000.)\n    cfg.commands.base_velocity.heading_command = False\n    cfg.commands.base_velocity.rel_standing_envs = 0.\n    with launch_simulation(cfg, {\'visualizer\': None, \'visualizer_explicit\': True}):\n        raw = gym.make(args.task, cfg=cfg)\n        env = RslRlVecEnvWrapper(raw, clip_actions=agent_cfg.clip_actions)\n        runner = OnPolicyRunner(env, agent_cfg.to_dict(), log_dir=None, device=agent_cfg.device)\n        runner.load(args.checkpoint)\n        policy = runner.get_inference_policy(device=env.unwrapped.device)\n        base = env.unwrapped\n        command = base.command_manager.get_term(\'base_velocity\')\n        robot = base.scene[\'robot\']\n        dt = base.step_dt\n        alive = torch.ones(5, dtype=torch.bool, device=base.device)\n        survival = torch.full((5,), 10., device=base.device)\n        error = torch.zeros(5, device=base.device)\n        count = torch.zeros(5, device=base.device)\n        phase_velocity = torch.zeros((4, 5, 3), device=base.device)\n        phase_count = torch.zeros((4, 5), device=base.device)\n        initial_xy = robot.data.root_pos_w[:, :2].clone()\n        path_length = torch.zeros(5, device=base.device)\n        with torch.inference_mode():\n            for step in range(round(10. / dt)):\n                t = step * dt\n                phase = 0 if t < 1 else (1 if t < 4 else (2 if t < 7 else 3))\n                target = (0., 0., 0.) if t < 1 else ((.2, 0., 0.) if t < 4 else ((0., .15, 0.) if t < 7 else (-.15, 0., 0.)))\n                command.vel_command_b[:] = torch.tensor(target, device=base.device)\n                # Refresh observations after setting the command, before inference.\n                obs = env.get_observations()\n                action = policy(obs)\n                error += torch.linalg.vector_norm(robot.data.root_lin_vel_b[:, :2] - command.vel_command_b[:, :2], dim=1) * alive\n                count += alive\n                phase_velocity[phase, :, :2] += robot.data.root_lin_vel_b[:, :2] * alive[:, None]\n                phase_velocity[phase, :, 2] += robot.data.root_ang_vel_b[:, 2] * alive\n                phase_count[phase] += alive\n                before_xy = robot.data.root_pos_w[:, :2].clone()\n                obs, _, dones, _ = env.step(action)\n                path_length += torch.linalg.vector_norm(robot.data.root_pos_w[:, :2] - before_xy, dim=1) * (alive & ~dones.bool())\n                newly_done = alive & dones.bool()\n                survival[newly_done] = (step + 1) * dt\n                alive &= ~dones.bool()\n                policy.reset(dones)\n        result = {\n            \'checkpoint\': args.checkpoint, \'seed\': cfg.seed,\n            \'protocol\': \'Five nominal trials: stand 1s, forward 3s, lateral 3s, backward 3s. Stop scoring each trial at its first termination.\',\n            \'survived_10_seconds\': int(alive.sum()), \'trials\': 5,\n            \'survival_seconds\': survival.cpu().tolist(),\n            \'mean_xy_velocity_error_m_s\': (error / count.clamp_min(1)).cpu().tolist(),\n            \'stationary_reference_mean_xy_error_m_s\': 0.15,\n            \'phase_names\': [\'stand\', \'forward\', \'lateral\', \'backward\'],\n            \'phase_command_vx_vy_wz\': [[0., 0., 0.], [.2, 0., 0.], [0., .15, 0.], [-.15, 0., 0.]],\n            \'phase_mean_actual_vx_vy_wz_per_trial\': (phase_velocity / phase_count.clamp_min(1)[:, :, None]).cpu().tolist(),\n            \'xy_path_length_m\': path_length.cpu().tolist(),\n            \'net_xy_displacement_m_survivors\': torch.linalg.vector_norm(robot.data.root_pos_w[:, :2] - initial_xy, dim=1)[alive].cpu().tolist(),\n            \'box_transfer_success_established\': False,\n        }\n        Path(args.output).write_text(json.dumps(result, indent=2))\n        print(\'WALK_EVALUATION:\', json.dumps(result, indent=2), flush=True)\n        env.close()\n\n\nif __name__ == \'__main__\':\n    main()\n'
RUN_EVALUATION = False
if RUN_EVALUATION:
    import subprocess
    root = Path('/content/drive/MyDrive/LearningRL/zhaoyu-lu-HMD/rounds')
    folders = sorted([p for pattern in ('flat_walk_*', 'walk_tracking_*') for p in root.glob(pattern) if list(p.glob('model_*.pt'))], key=lambda p:tuple(p.name.split('_')[-2:]))
    assert folders, 'No walking checkpoint is backed up yet.'
    saved = folders[-1]
    checkpoint = max(saved.glob('model_*.pt'), key=lambda p:int(p.stem.split('_')[-1]))
    Path('/content/evaluate_walk.py').write_text(EVALUATION_SOURCE)
    with (saved/'evaluation.log').open('w') as log:
        result = subprocess.run([PYTHON, '/content/evaluate_walk.py', '--checkpoint', str(checkpoint), '--output', str(saved/'evaluation.json'), 'physics=newton_mjwarp'], cwd=LAB, env=env, stdout=log, stderr=subprocess.STDOUT, timeout=600)
    print((saved/'evaluation.log').read_text()[-6500:])
    result.check_returncode()
    print((saved/'evaluation.json').read_text())
else:
    print('Set RUN_EVALUATION=True to run five frozen-policy trials.')
